# Power your products with ChatGPT and your own data

This is a walkthrough taking readers through how to build starter Q&A and Chatbot applications using the ChatGPT API and their own data. 

It is laid out in these sections:
- **Setup:** 
    - Initiate variables and source the data
- **Lay the foundations:**
    - Set up the vector database to accept vectors and data
    - Load the dataset, chunk the data up for embedding and store in the vector database
- **Make it a product:**
    - Add a retrieval step where users provide queries and we return the most relevant entries
    - Summarise search results with GPT-3
    - Test out this basic Q&A app in Streamlit
- **Build your moat:**
    - Create an Assistant class to manage context and interact with our bot
    - Use the Chatbot to answer questions using semantic search context
    - Test out this basic Chatbot app in Streamlit
    
Upon completion, you have the building blocks to create your own production chatbot or Q&A application using OpenAI APIs and a vector database.

This notebook was originally presented with [these slides](https://drive.google.com/file/d/1dB-RQhZC_Q1iAsHkNNdkqtxxXqYODFYy/view?usp=share_link), which provide visual context for this journey.

In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup

First we'll setup our libraries and environment variables

In [24]:
import openai
import os
import requests
import numpy as np
import pandas as pd
from typing import Iterator
import tiktoken
import textract
from numpy import array, average

from database import get_redis_connection

# Set our default models and chunking size
from config import COMPLETIONS_MODEL, EMBEDDINGS_MODEL, CHAT_MODEL, TEXT_EMBEDDING_CHUNK_SIZE, VECTOR_FIELD_NAME

from transformers import init_openai_api

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ImportWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [25]:
pd.set_option('display.max_colwidth', 0)

In [26]:
data_dir = os.path.join(os.curdir,'data')
pdf_files = sorted([x for x in os.listdir(data_dir) if 'DS_Store' not in x])
pdf_files

["FIA Practice Directions - Competitor's Staff Registration System.pdf",
 'fia_2022_formula_1_sporting_regulations_-_issue_9_-_2022-10-19_0.pdf',
 'fia_2023_formula_1_technical_regulations_-_issue_4_-_2022-12-07.pdf',
 'fia_f1_power_unit_financial_regulations_issue_1_-_2022-08-16.pdf',
 'fia_formula_1_financial_regulations_iss.13.pdf']

## Laying the foundations

### Storage

We're going to use Redis as our database for both document contents and the vector embeddings. You will need the full Redis Stack to enable use of Redisearch, which is the module that allows semantic search - more detail is in the [docs for Redis Stack](https://redis.io/docs/stack/get-started/install/docker/).

To set this up locally, you will need to install Docker and then run the following command: ```docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest```.

The code used here draws heavily on [this repo](https://github.com/RedisAI/vecsim-demo).

After setting up the Docker instance of Redis Stack, you can follow the below instructions to initiate a Redis connection and create a Hierarchical Navigable Small World (HNSW) index for semantic search.

In [52]:
# Setup Redis
from redis import Redis
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField,
    NumericField
)
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)

redis_client = get_redis_connection(host='openai-capps-vesjek-redis.yellowdune-77a2352f.eastus.azurecontainerapps.io')

In [53]:
# Constants
VECTOR_DIM = 1536 #len(data['title_vector'][0]) # length of the vectors
#VECTOR_NUMBER = len(data)                 # initial number of vectors
PREFIX = "sportsdoc"                            # prefix for the document keys
DISTANCE_METRIC = "COSINE"                # distance metric for the vectors (ex. COSINE, IP, L2)

In [54]:
# Create search index

# Index
INDEX_NAME = "f1-index"           # name of the search index
VECTOR_FIELD_NAME = 'content_vector'

# Define RediSearch fields for each of the columns in the dataset
# This is where you should add any additional metadata you want to capture
filename = TextField("filename")
text_chunk = TextField("text_chunk")
file_chunk_index = NumericField("file_chunk_index")

# define RediSearch vector fields to use HNSW index

text_embedding = VectorField(VECTOR_FIELD_NAME,
    "HNSW", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC
    }
)
# Add all our field objects to a list to be created as an index
fields = [filename,text_chunk,file_chunk_index,text_embedding]

In [55]:
redis_client.ping()

True

In [56]:
# Optional step to drop the index if it already exists
#redis_client.ft(INDEX_NAME).dropindex()

# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except Exception as e:
    print(e)
    # Create RediSearch Index
    print('Not there yet. Creating')
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

Index already exists


### Ingestion

We'll load up our PDFs and do the following
- Initiate our tokenizer
- Run a processing pipeline to:
    - Mine the text from each PDF
    - Split them into chunks and embed them
    - Store them in Redis

In [57]:
# The transformers.py file contains all of the transforming functions, including ones to chunk, embed and load data
# For more details, check the file and work through each function individually
from transformers import handle_file_string

In [58]:
%%time
# This step takes about 5 minutes

init_openai_api()

# Initialise tokenizer
tokenizer = tiktoken.get_encoding("cl100k_base")

init_openai_api()

# Process each PDF file and prepare for embedding
#for pdf_file in pdf_files:
    
#    pdf_path = os.path.join(data_dir,pdf_file)
#    print(pdf_path)
    
    # Extract the raw text from each PDF using textract
#    text = textract.process(pdf_path, method='pdfminer')
    
    # Chunk each document, embed the contents and load to Redis
#    handle_file_string((pdf_file,text.decode("utf-8")),tokenizer,redis_client,VECTOR_FIELD_NAME,INDEX_NAME)

CPU times: user 36 µs, sys: 7 µs, total: 43 µs
Wall time: 48.6 µs


In [34]:
# Check that our docs have been inserted
redis_client.ft(INDEX_NAME).info()['num_docs']

'660'

## Make it a product

Now we can test that our search works as intended by:
- Querying our data in Redis using semantic search and verifying results
- Adding a step to pass the results to GPT-3 for summarisation

In [35]:
from database import get_redis_results

In [59]:
%%time

f1_query='member of staff minimum'

result_df = get_redis_results(redis_client,f1_query,index_name=INDEX_NAME)
result_df.head(2)

CPU times: user 30.7 ms, sys: 3.8 ms, total: 34.5 ms
Wall time: 1.6 s


,id,result,certainty
0,0,"REGISTERED MEMBERS OF COMPETITOR’ STAFF A minimum of 6 persons shall in principle be registered per Competitor:  Team Principal: the competitor’s most important decision-maker  Sporting Director: ensures the compliance of the Competitor with the sporting regulations  Technical Director: ensures the compliance of the Competitor with the technical regulations  Team Manager: holds the operational responsibility of the Competitor at the events  2 Race Engineers responsible for the Competitor's cars. The name of a member of staff may appear in two places on the Competitor registration form, on condition that the relevant Competitor justifies to the satisfaction of the FIA, by producing its organisational chart, that the individual concerned is indeed performing these two sets of duties. In that event an individual would only need one registration. 1 / 5 FIA Legal Department Practice Directions - Competitor’s Staff Registration System 17 March 2011 III. REGISTRATION PROCESS In the FIA World Championships, it is the responsibility of the Competitors to communicate to the FIA the list of the members of their staff who are to be registered as Competitor's Staff. Each duly registered member of a Competitor's staff will be given, via the Competitor, a Competitor’s Staff Certificate of Registration with the FIA (see Appendix 4), a document that is drawn up and issued by the FIA and that remains the property of the FIA. (i) Entry closing date: When applying to enter an FIA World Championship, the Competitor must communicate to the FIA the list of the members of their staff who are to be registered as Competitor's Staff by providing the FIA with the form drawn up specifically for that purpose (see Appendix 4), duly signed and completed.",0.1883482337
1,1,"(ii) Renewal date: 1st of January of each year (iii) Modification of the list of members of the staff: The Competitor must inform the FIA of this within the 7 days following that change and submit an updated list within the same deadline, while returning to the FIA the Competitor’s Staff Certificates of Registration of the persons who have ceased to perform their role. The FIA will follow the same procedure for applications during the course of a season as that applicable for annual applications. The FIA will not charge a fee for issuing or renewing a Competitor’s Staff Certificate of Registration. IV. REFUSAL TO ISSUE A COMPETITOR’S STAFF CERTIFICATE OF REGISTRATION (see applicable procedure in Appendix 1) The FIA reserves the right to refuse to issue a Competitor’s Staff Certificate of Registration to any person who is under a disciplinary sanction imposed by the IT and/or the ICA or is in breach of the FIA Code of Good Standing (see Appendix B to the ISC). If the FIA, after having received an application for a Competitor’s Staff Certificate of Registration, intends to refuse to grant a Competitor’s Staff Certificate of Registration, it will, within 8 days after the receipt of such application, inform the relevant person and Competitor of such proposed refusal, giving its written detailed reasons, and then give the person and the Competitor an opportunity to make representations to it in writing (including by email) regarding such refusal within 8 days after having received notification of the intended refusal.",0.21464407444


In [60]:
# Build a prompt to provide the original query, the result and ask to summarise for the user
summary_prompt = '''Summarise this result in a bulleted list to answer the search query a customer has sent.
Search query: SEARCH_QUERY_HERE
Search result: SEARCH_RESULT_HERE
Summary:
'''
summary_prepped = summary_prompt.replace('SEARCH_QUERY_HERE',f1_query).replace('SEARCH_RESULT_HERE',result_df['result'][0])
summary = openai.Completion.create(engine=COMPLETIONS_MODEL,prompt=summary_prepped,max_tokens=500)
# Response provided by GPT-3
print(summary['choices'][0]['text'])

- Minimum of 6 persons must be registered per Competitor
- Includes Team Principal, Sporting Director, Technical Director, Team Manager, 2 Race Engineers 
- Name of a member of staff may appear on registration form twice, if approved by FIA
- Competitors must communicate list of registered staff to FIA
- FIA will issue Competitor's Staff Certificate of Registration to each registered member


### Search

Now that we've got our knowledge embedded and stored in Redis, we can now create an internal search application. Its not sophisticated but it'll get the job done for us.

In the directory containing this app, execute ```streamlit run search.py```. This will open up a Streamlit app in your browser where you can ask questions of your embedded data.

__Example Questions__:
- what is the cost cap for a power unit in 2023
- what should competitors include on their application form

## Build your moat

The Q&A was useful, but fairly limited in the complexity of interaction we can have - if the user asks a sub-optimal question, there is no assistance from the system to prompt them for more info or conversation to lead them down the right path.

For the next step we'll make a Chatbot using the Chat Completions endpoint, which will:
- Be given instructions on how it should act and what the goals of its users are
- Be supplied some required information that it needs to collect
- Go back and forth with the customer until it has populated that information
- Say a trigger word that will kick off semantic search and summarisation of the response

For more details on our Chat Completions endpoint and how to interact with it, please check out the docs [here](https://platform.openai.com/docs/guides/chat).

### Framework

This section outlines a basic framework for working with the API and storing context of previous conversation "turns". Once this is established, we'll extend it to use our retrieval endpoint.

In [61]:
from config import AZURE_OPENAI_API_KEY, AZURE_OPENAI_BASE_URL, AZURE_OPENAI_CHAT_DEPLOYMENT_NAME, AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME

openai.api_version = '2023-05-15'
openai.api_type = 'azure'
openai.api_key = AZURE_OPENAI_API_KEY
openai.api_base = AZURE_OPENAI_BASE_URL

# A basic example of how to interact with our ChatCompletion endpoint
# It requires a list of "messages", consisting of a "role" (one of system, user or assistant) and "content"
question = 'How can you help me'


completion = openai.ChatCompletion.create(
  deployment_id=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
  model="gpt-3.5-turbo",
  messages=[
    {"role": "user", "content": question}
  ]
)
print(f"{completion['choices'][0]['message']['role']}: {completion['choices'][0]['message']['content']}")

assistant: As an AI language model, I can help you in various ways depending on your needs. I can assist you in writing documents, proofreading, editing, answering questions, providing information and much more. You can give me a command or ask me a question, and I will do my best to provide you with the information or assistance you require.


In [62]:
from termcolor import colored

# A basic class to create a message as a dict for chat
class Message:
    
    
    def __init__(self,role,content):
        
        self.role = role
        self.content = content
        
    def message(self):
        
        return {"role": self.role,"content": self.content}
        
# Our assistant class we'll use to converse with the bot
class Assistant:
    
    def __init__(self):
        self.conversation_history = []

    def _get_assistant_response(self, prompt):
        
        try:
            completion = openai.ChatCompletion.create(
              deployment_id=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
              model="gpt-3.5-turbo",
              messages=prompt
            )
            
            response_message = Message(completion['choices'][0]['message']['role'],completion['choices'][0]['message']['content'])
            return response_message.message()
            
        except Exception as e:
            
            return f'Request failed with exception {e}'

    def ask_assistant(self, next_user_prompt, colorize_assistant_replies=True):
        [self.conversation_history.append(x) for x in next_user_prompt]
        assistant_response = self._get_assistant_response(self.conversation_history)
        self.conversation_history.append(assistant_response)
        return assistant_response
            
        
    def pretty_print_conversation_history(self, colorize_assistant_replies=True):
        for entry in self.conversation_history:
            if entry['role'] == 'system':
                pass
            else:
                prefix = entry['role']
                content = entry['content']
                output = colored(prefix +':\n' + content, 'green') if colorize_assistant_replies and entry['role'] == 'assistant' else prefix +':\n' + content
                print(output)

In [63]:
# Initiate our Assistant class
conversation = Assistant()

# Create a list to hold our messages and insert both a system message to guide behaviour and our first user question
messages = []
system_message = Message('system','You are a helpful business assistant who has innovative ideas')
user_message = Message('user','What can you do to help me')
messages.append(system_message.message())
messages.append(user_message.message())
messages

[{'role': 'system',
  'content': 'You are a helpful business assistant who has innovative ideas'},
 {'role': 'user', 'content': 'What can you do to help me'}]

In [64]:
# Get back a response from the Chatbot to our question
response_message = conversation.ask_assistant(messages)

response_message

print(response_message['content'])

As a business assistant, I can help you in a number of ways. Here are some innovative ideas to assist you in your business:

1. Streamline your workflow: I can help you identify inefficiencies in your business process and suggest new strategies to improve productivity and effectiveness.

2. Introduce new marketing strategies: I can help you identify new marketing channels and approaches to reach new customers, increase brand awareness, and ensure customer retention.

3. Create better customer experiences: I can help you improve your customer services and experience through the use of new technologies and innovative tactics.

4. Improve your financial management: I can help you track your financials more effectively, identify areas where you are spending too much, and make suggestions for managing your expenses more efficiently.

5. Develop new product ideas: I can help you brainstorm new product ideas, conduct market research, and develop solutions that meet the needs of your customers

In [65]:
next_question = 'Tell me more about option 2'

# Initiate a fresh messages list and insert our next question
messages = []
user_message = Message('user',next_question)
messages.append(user_message.message())
response_message = conversation.ask_assistant(messages)
print(response_message['content'])

Marketing is a crucial aspect of any business, and it is important to continually explore new channels and approaches to attract and retain customers. Here are some specific ideas that I could help you develop to improve your marketing:

1. Social Media Marketing: We could explore new social media platforms that are popular amongst your target audience, design engaging content and develop a social media strategy that will help you reach more people, engage better, and drive conversions.

2. Influencer Marketing: We could identify key influencers in your industry or among your target audience and develop partnerships or influencer marketing campaigns to increase brand awareness and reach more potential customers.

3. Video Marketing: Videos are increasingly becoming a popular tool to convey your message in a creative and interactive way. We could create engaging videos that offer value to your audience, showcase your products or services, and help develop your brand identity.

4. Conten

In [66]:
# Print out a log of our conversation so far

conversation.pretty_print_conversation_history()

user:
What can you do to help me
assistant:
As a business assistant, I can help you in a number of ways. Here are some innovative ideas to assist you in your business:

1. Streamline your workflow: I can help you identify inefficiencies in your business process and suggest new strategies to improve productivity and effectiveness.

2. Introduce new marketing strategies: I can help you identify new marketing channels and approaches to reach new customers, increase brand awareness, and ensure customer retention.

3. Create better customer experiences: I can help you improve your customer services and experience through the use of new technologies and innovative tactics.

4. Improve your financial management: I can help you track your financials more effectively, identify areas where you are spending too much, and make suggestions for managing your expenses more efficiently.

5. Develop new product ideas: I can help you brainstorm new product ideas, conduct market research, and develop sol

### Knowledge retrieval

Now we'll extend the class to call a downstream service when a stop sequence is spoken by the Chatbot.

The main changes are:
- The system message is more comprehensive, giving criteria for the Chatbot to advance the conversation
- Adding an explicit stop sequence for it to use when it has the info it needs
- Extending the class with a function ```_get_search_results``` which sources Redis results

In [67]:
# Updated system prompt requiring Question and Year to be extracted from the user
system_prompt = '''
You are a helpful Formula 1 knowledge base assistant. You need to capture a Question and Year from each customer.
The Question is their query on Formula 1, and the Year is the year of the applicable Formula 1 season.
If they haven't provided the Year, ask them for it again.
Once you have the Year, say "searching for answers".

Example 1:

User: I'd like to know the cost cap for a power unit

Assistant: Certainly, what year would you like this for?

User: 2023 please.

Assistant: Searching for answers.
'''

# New Assistant class to add a vector database call to its responses
class RetrievalAssistant:
    
    def __init__(self):
        self.conversation_history = []  

    def _get_assistant_response(self, prompt):
        
        try:
            completion = openai.ChatCompletion.create(
              deployment_id=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
              model=CHAT_MODEL,
              messages=prompt,
              temperature=0.1
            )
            
            response_message = Message(completion['choices'][0]['message']['role'],completion['choices'][0]['message']['content'])
            return response_message.message()
            
        except Exception as e:
            
            return f'Request failed with exception {e}'
    
    # The function to retrieve Redis search results
    def _get_search_results(self,prompt):
        latest_question = prompt
        search_content = get_redis_results(redis_client,latest_question,INDEX_NAME)['result'][0]
        return search_content
        

    def ask_assistant(self, next_user_prompt):
        [self.conversation_history.append(x) for x in next_user_prompt]
        assistant_response = self._get_assistant_response(self.conversation_history)
        
        # Answer normally unless the trigger sequence is used "searching_for_answers"
        if 'searching for answers' in assistant_response['content'].lower():
            question_extract = openai.Completion.create(
                deployment_id='text-davinci-003',
                model=COMPLETIONS_MODEL,prompt=f"Extract the user's latest question and the year for that question from this conversation: {self.conversation_history}. Extract it as a sentence stating the Question and Year")
            search_result = self._get_search_results(question_extract['choices'][0]['text'])
            
            # We insert an extra system prompt here to give fresh context to the Chatbot on how to use the Redis results
            # In this instance we add it to the conversation history, but in production it may be better to hide
            self.conversation_history.insert(-1,{"role": 'system',"content": f"Answer the user's question using this content: {search_result}. If you cannot answer the question, say 'Sorry, I don't know the answer to this one'"})
            #[self.conversation_history.append(x) for x in next_user_prompt]
            
            assistant_response = self._get_assistant_response(self.conversation_history)
            print(next_user_prompt)
            print(assistant_response)
            self.conversation_history.append(assistant_response)
            return assistant_response
        else:
            self.conversation_history.append(assistant_response)
            return assistant_response
            
        
    def pretty_print_conversation_history(self, colorize_assistant_replies=True):
        for entry in self.conversation_history:
            if entry['role'] == 'system':
                pass
            else:
                prefix = entry['role']
                content = entry['content']
                output = colored(prefix +':\n' + content, 'green') if colorize_assistant_replies and entry['role'] == 'assistant' else prefix +':\n' + content
                #prefix = entry['role']
                print(output)

In [68]:
conversation = RetrievalAssistant()
messages = []
system_message = Message('system',system_prompt)
user_message = Message('user','How can a competitor be disqualified from competition')
messages.append(system_message.message())
messages.append(user_message.message())
response_message = conversation.ask_assistant(messages)
response_message

{'role': 'assistant', 'content': 'Sure, what year would you like this for?'}

In [69]:
messages = []
user_message = Message('user','For 2023 please.')
messages.append(user_message.message())
response_message = conversation.ask_assistant(messages)
#response_message

[{'role': 'user', 'content': 'For 2023 please.'}]
{'role': 'assistant', 'content': 'According to the FIA Sporting Regulations for the 2023 Formula One season, a competitor can be disqualified from the competition if they breach any of the FIA regulations. The International Tribunal (IT) will investigate and establish the existence of the breach and impose any sanction upon the person and competitor concerned. The President of the FIA, in its capacity as prosecuting authority, will ask for the imposition of a suspension upon Competitor’s Staff Certificate of Registration holders who have contravened the FIA Code of Good Standing or the withdrawal of the Competitor’s Staff Certificate of Registration. The person and/or competitor sanctioned may bring an appeal before the ICA against the IT’s decision.'}


In [70]:
conversation.pretty_print_conversation_history()

user:
How can a competitor be disqualified from competition
assistant:
Sure, what year would you like this for?
user:
For 2023 please.
assistant:
According to the FIA Sporting Regulations for the 2023 Formula One season, a competitor can be disqualified from the competition if they breach any of the FIA regulations. The International Tribunal (IT) will investigate and establish the existence of the breach and impose any sanction upon the person and competitor concerned. The President of the FIA, in its capacity as prosecuting authority, will ask for the imposition of a suspension upon Competitor’s Staff Certificate of Registration holders who have contravened the FIA Code of Good Standing or the withdrawal of the Competitor’s Staff Certificate of Registration. The person and/or competitor sanctioned may bring an appeal before the ICA against the IT’s decision.


### Chatbot

Now we'll put all this into action with a real (basic) Chatbot.

In the directory containing this app, execute ```streamlit run chat.py```. This will open up a Streamlit app in your browser where you can ask questions of your embedded data. 

__Example Questions__:
- what is the cost cap for a power unit in 2023
- what should competitors include on their application form
- how can a competitor be disqualified

### Consolidation

Over the course of this notebook you have:
- Laid the foundations of your product by embedding our knowledge base
- Created a Q&A application to serve basic use cases
- Extended this to be an interactive Chatbot

These are the foundational building blocks of any Q&A or Chat application using our APIs - these are your starting point, and we look forward to seeing what you build with them!